<a href="https://colab.research.google.com/github/Maddox159-crypto/ESAA_assignment/blob/main/%ED%95%84%EC%82%AC26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[개념]

```


1.   TF-IDF로 문서를 숫자로 변환하기  
: K-Means는 텍스트 자체를 이해하지 못하기에 문서를 숫자 벡터로 변환해야 함.


*   코드 예시  
tfidf_vect = TfidfVectorizer(
    tokenizer=LemNormalize,
    stop_words='english',
    ngram_range=(1,2),
    min_df=0.05,
    max_df=0.85
)
*   tokenizer = LemNormalize  
단어를 정규화 및 표제어 추출(lemmatization)해서 비슷한 형태의 단어를 서로 통일함. (예: cars를 car로 통일)  


*   stop_cords = 'english'  
: the, is, a 같은 영어 불용어 제거용  
*   ngram_range=(1, 2)  
: 1개짜리 단어 및 2개짜리 단어 조합 모두 사용하게 할 수 있음,
-> battery, life, battery life 이 3단어를 각각 별도의 피처로 인식하게 할 수 있음. (battery lie is good-> battery, life, battery life, good, life good), 근데 컴퓨터가 이걸 알고 조합을 만드는게 아니라 무의미하게 만드는 거임.


*   min_df=0.05  / max_df= 0.85
: 전체 문서 중 너무 적게/많이 등장하는 단어 제거
*  변환:  
feature_vect = tfidf_vect.fit_transform(
    document_df['opinion_text']
)


2.   K-Means로 문서 군집화


*   코드 예시  
km_cluster = KMeans(
    n_clusters=5,
    max_iter=10000,
    random_state=0
)
*   n_clusters=5  
: 문서를 5개 그룹으로 나누기  


*   max_iters=10000  
: 중심점 업데이트를 최대 10000번까지  

결론-- 붙어있는 단어 조합 다 만들기-> max(min)_df로 너무 흔함/레어한 조합 제거 -> TF-IDF로 각 표현의 중요도를 계산함-> 그러면 무지성으로 단어를 초반에 조합하더라도 나중에는 의미있는(실제로 자주 등장하는) 표현이 피처로 살아남게됨.  
cf) k값이 너무 크면 같은 주제가 여러 군집으로 나뉘어질 수 있으며, 너무 작으면 서로 다른 주제가 하나로 합쳐질 수 있음.





3.   군집 중심 centroid  
: K-Means에는 각 군집마다 중심점(centroid)가 존재하는데, centroid값이 큰 단어일수록 해당 군집의 대표단어이므로, 해당 군집에서 가장 중요한 n 개 선택해서 단어 이름을 가져옴.(top n)<- 각 군집의 특성을 해석하려고 사용


*   cluster_centers = km_cluster.cluster_centers_  
(3, 4611) = 군집 개수 3개, TF-IDF 단어 피처 4611개( 각 군집마다 4611단어에 대한 중심값이 존재한다는 의미임)
*  cluster_centers_[0, 1]  
= cluster0에서 두번째 단어 피처가 가지는 중심값.

```

[필사]

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import nltk

from nltk.tokenize import word_tokenize

from nltk.stem import WordNetLemmatizer
nltk.download('punkt')

nltk.download('punkt_tab')

nltk.download('wordnet')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [10]:
import pandas as pd
import glob, os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 700)
path = '/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1.0/topics'
all_files = glob.glob(os.path.join(path, "*.data"))
filename_list = []
opinion_text = []
for file_ in all_files:
    df = pd.read_table(file_, index_col=None, header=0, encoding='latin1')
    filename_ = file_.split('\\')[-1]
    filename = filename_.split('.')[0]
    filename_list.append(filename)
    opinion_text.append(df.to_string())
document_df = pd.DataFrame({'filename':filename_list, 'opinion_text':opinion_text})
document_df.head()


,filename,opinion_text
0,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"3 quot widescreen display was a bonus .\n0 This made for smoother graphics on the 255w of the vehicle moving along displayed roads, where the 750's display was more of a jerky movement .\n1 ..."
1,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,The food for our event was delicious .\n0 ...
2,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,You also get upscale features like spoken directions including street names and programmable POIs .\n0 I used to hesitate to go out of my directions but no...
3,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# .\n1 Great gas mileage and comfortable on long trips ..."
4,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ..."


In [13]:
def LemNormalize(text):

    lemmatizer = WordNetLemmatizer()

    return [lemmatizer.lemmatize(token) for token in word_tokenize(text)]

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english',\
                             ngram_range=(1,2), min_df=0.05, max_df=0.85)
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

In [15]:
from sklearn.cluster import KMeans
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_
cluster_centers = km_cluster.cluster_centers_

In [16]:
document_df['cluster_label'] = cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"3 quot widescreen display was a bonus .\n0 This made for smoother graphics on the 255w of the vehicle moving along displayed roads, where the 750's display was more of a jerky movement .\n1 ...",2
1,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,The food for our event was delicious .\n0 ...,0
2,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,You also get upscale features like spoken directions including street names and programmable POIs .\n0 I used to hesitate to go out of my directions but no...,2
3,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# .\n1 Great gas mileage and comfortable on long trips ...",4
4,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",0


In [17]:
document_df[document_df['cluster_label']==0].sort_values(by='filename')

,filename,opinion_text,cluster_label
1,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,The food for our event was delicious .\n0 ...,0
4,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",0
9,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room .\n0 They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day .\n1 ...,0
19,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,The room was packed to capacity with queues at the food buffets .\n0 The over zealous staff cleared our unfinished drinks while we were collecting cooked food and movement around the room with plates was difficult in the crowded circumstances .\n1 ...,0
20,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Good Value good location , ideal choice .\n0 Great Location , Nice Rooms , Helpless Concierge\n1 ...",0
23,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",0
24,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"not customer, oriented hotelvery low service levelboor reception\n0 The room was quiet, clean, the bed and pillows were comfortable, and the serv...",0
26,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city .\n0 The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio .\n1 ...",0
27,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,Mediocre room and service for a very extravagant price .\n0 ...,0
28,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Both of us having worked in tourism for over 14 years were very disappointed at the level of service provided by this gentleman .\n0 The service was good, very friendly staff and we loved the free wine reception each night .\n1 ...",0


In [18]:
document_df[document_df['cluster_label']==1].sort_values(by='filename')

,filename,opinion_text,cluster_label
11,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 ...,1
22,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Very happy with my 08 Accord, performance is quite adequate it has nice looks and is a great long, distance cruiser .\n0 6, 4, 3 eco engine has poor performance and gas mileage of 22 highway .\n1 Overall performance is good but comfort level is poor .\n2 ...",1
32,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"It's quiet, get good gas mileage and looks clean inside and out .\n0 The mileage is great, and I've had to get used to stopping less for gas .\n1 Thought gas ...",1
47,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"After slowing down, transmission has to be kicked to speed up .\n0 ...",1


In [19]:
document_df[document_df['cluster_label']==2].sort_values(by='filename')

,filename,opinion_text,cluster_label
0,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"3 quot widescreen display was a bonus .\n0 This made for smoother graphics on the 255w of the vehicle moving along displayed roads, where the 750's display was more of a jerky movement .\n1 ...",2
45,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,Another feature on the 255w is a display of the posted speed limit on the road which you are currently on right above your current displayed speed .\n0 I found myself not even looking at my car speedometer as I could easily see my current speed and the speed limit of my route at a glance .\n1 ...,2
44,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Another thing to consider was that I paid $50 less for the 750 and it came with the FM transmitter cable and a USB cord to connect it to your computer for updates and downloads .\n0 update and reroute much _more_ quickly than my other GPS .\n1 UPDATE ON THIS , It finally turned out that to see the elevation contours at lowe...",2
42,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Windows 7 is quite simply faster, more stable, boots faster, goes to sleep faster, comes back from sleep faster, manages your files better and on top of that it's beautiful to look at and easy to use .\n0 , faster about 20% to 30% faster at running applications than my Vista , seriously\n1 ...",2
40,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"As always, the video screen is sharp and bright .\n0 2, inch screen and a glossy, polished aluminum finish that one CNET editor described as looking like a Christmas tree ornament .\n1 ...",2
39,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"A few other things I'd like to point out is that you must push the micro, sized right angle end of the ac adapter until it snaps in place or the battery may not charge .\n0 The full size right shift k...",2
36,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"It's fast to acquire satellites .\n0 If you've ever had a Brand X GPS take you on some strange route that adds 20 minutes to your trip, has you turn the wrong way down a one way road, tell you to turn AFTER you've passed the street, frequently loses the satellite signal, or has old maps missing streets, you know how important this stuff is .\n1 ...",2
34,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,Keep in mind that once you get in a room full of light or step outdoors screen reflections could become annoying .\n0 I've used mine outsi...,2
33,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,It is easy to read and when touching the screen it works great !\n0 and zoom out buttons on the 255w to the same side of the screen which makes it a bit easier .\n1 ...,2
31,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"In fact, the entire navigation structure has been completely revised , I'm still getting used to it but it's a huge step forward .\n0 ...",2


In [20]:
document_df[document_df['cluster_label']==3].sort_values(by='filename')

,filename,opinion_text,cluster_label
7,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,short battery life I moved up from an 8gb .\n0 I love this ipod except for the battery life .\n1 ...,3
8,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh Li, ion Battery , and a 1 .\n0 Not to mention that as of now...",3
13,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"I thought it would be fitting to christen my Kindle with the Stephen King novella UR, so went to the Amazon site on my computer and clicked on the button to buy it .\n0 As soon as I'd clicked the button to confirm my order it appeared on my Kindle almost immediately !\n1 ...",3
17,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"After I plugged it in to my USB hub on my computer to charge the battery the charging cord design is very clever !\n0 After you have paged tru a 500, page book one, page, at, a, time to get from Chapter 2 to Chapter 15, see how excited you are about a low battery and all the time it took to get there !\n1 ...",3
37,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"If a case was included, as with the Kindle 1, that would have been reflected in a higher price .\n0 lower overall price, with nice leather cover .\n1 ...",3
38,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"The Eee Super Hybrid Engine utility lets users overclock or underclock their Eee PC's to boost performance or provide better battery life depending on their immediate requirements .\n0 In Super Performance mode CPU, Z shows the bus speed to increase up to 169 .\n1 One...",3
43,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,headphone jack i got a clear case for it and it i got a clear case for it and it like prvents me from being able to put the jack all the way in so the sound can b messsed up or i can get it in there and its playing well them go to move or something and it slides out .\n0 Picture and sound quality are excellent for this typ of devic .\n1 ...,3


In [21]:
document_df[document_df['cluster_label']==4].sort_values(by='filename')

,filename,opinion_text,cluster_label
3,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Ride seems comfortable and gas mileage fairly good averaging 26 city and 30 open road .\n0 Seats are fine, in fact of all the smaller sedans this is the most comfortable I found for the price as I am 6', 2 and 250# .\n1 Great gas mileage and comfortable on long trips ...",4
5,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,I love the new body style and the interior is a simple pleasure except for the center dash .\n0 ...,4
6,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"First of all, the interior has way too many cheap plastic parts like the cheap plastic center piece that houses the clock .\n0 3 blown struts at 30,000 miles, interior trim coming loose and rattling squeaking, stains on paint, and bug splats taking paint off, premature uneven brake wear, on 3rd windsh...",4
14,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Drivers seat not comfortable, the car itself compared to other models of similar class .\n0 ...",4
21,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,I previously owned a Toyota 4Runner which had incredible build quality and reliability .\n0 I bought the Camry because of Toyota reliability and qua...,4
25,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Front seats are very uncomfortable .\n0 No memory seats, no trip computer, can only display outside temp with trip odometer .\n1 ...",4


In [22]:
from sklearn.cluster import KMeans
km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_
document_df['cluster_label']=cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
1,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,The food for our event was delicious .\n0 ...,0
4,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"The room was not overly big, but clean and very comfortable beds, a great shower and very clean bathrooms .\n0 The second room was smaller, with a very inconvenient bathroom layout, but at least it was quieter and we were able to sleep .\n1 ...",0
9,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,The wine reception is a great idea as it is nice to meet other travellers and great having access to the free Internet access in our room .\n0 They also have a computer available with free internet which is a nice bonus but I didn't find that out till the day before we left but was still able to get on there to check our flight to Vegas the next day .\n1 ...,0
28,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Both of us having worked in tourism for over 14 years were very disappointed at the level of service provided by this gentleman .\n0 The service was good, very friendly staff and we loved the free wine reception each night .\n1 ...",0
30,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,Parking was expensive but I think this is common for San Fran .\n0 there is a fee for parking but well worth it seeing no where to park if you do have a car .\n1 ...,0
23,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"All in all, a normal chain hotel on a nice location , I will be back if I do not find anthing closer to Picadilly for a better price .\n0 ...",0
20,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"Good Value good location , ideal choice .\n0 Great Location , Nice Rooms , Helpless Concierge\n1 ...",0
29,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"We arrived at 23,30 hours and they could not recommend a restaurant so we decided to go to Tesco, with very limited choices but when you are hingry you do not careNext day they rang the bell at 8,00 hours to clean the room, not being very nice being waken up so earlyEvery day they gave u...",0
24,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"not customer, oriented hotelvery low service levelboor reception\n0 The room was quiet, clean, the bed and pillows were comfortable, and the serv...",0
26,/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1,"The Swissotel is one of our favorite hotels in Chicago and the corner rooms have the most fantastic views in the city .\n0 The rooms look like they were just remodled and upgraded, there was an HD TV and a nice iHome docking station to put my iPod so I could set the alarm to wake up with my music instead of the radio .\n1 ...",0


In [23]:
cluster_centers = km_cluster.cluster_centers_
print('cluster_centers shape :', cluster_centers.shape)
print(cluster_centers)

cluster_centers shape : (3, 6155)
[[0.         0.00040956 0.00090396 ... 0.00175541 0.00139053 0.00139053]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.00163124 0.00119946 0.         ... 0.         0.         0.        ]]


In [31]:
def get_cluster_details(cluster_model, cluster_data, feature_names, clusters_num,
                        top_n_features=10):

    cluster_details = {}

    centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:,::-1]

    for cluster_num in range(cluster_model.n_clusters):
        cluster_details[cluster_num]={}
        cluster_details[cluster_num]['cluster']=cluster_num

        top_feature_indexes = centroid_feature_ordered_ind[cluster_num, :top_n_features]
        top_features = [feature_names[ind] for ind in top_feature_indexes]

        top_feature_values = cluster_model.cluster_centers_[cluster_num, top_feature_indexes].tolist()
        cluster_details[cluster_num]['top_features'] = top_features
        cluster_details[cluster_num]['top_features_value'] = top_feature_values
        filenames = cluster_data[cluster_data['cluster_label']==cluster_num]['filename']
        filenames = filenames.values.tolist()
        cluster_details[cluster_num]['filenames']=filenames

    return cluster_details

In [32]:
def print_cluster_details(cluster_details):
    for cluster_num, cluster_detail in cluster_details.items():
        print('####### Cluster {0}'.format(cluster_num))
        print('Top features:', cluster_detail['top_features'])
        print('Reviews file names :',cluster_detail['filenames'][:7])
        print('==================================================')

In [33]:
feature_names = tfidf_vect.get_feature_names_out()
cluster_details = get_cluster_details(cluster_model=km_cluster, cluster_data=document_df,
                                      feature_names = feature_names, clusters_num=3, top_n_features=10)
print_cluster_details(cluster_details)

####### Cluster 0
Top features: ['room', 'hotel', 'service', 'staff', 'food', 'location', 'bathroom', 'clean', 'price', 'parking']
Reviews file names : ['/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1', '/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1', '/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1', '/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1', '/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1', '/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1', '/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1']
####### Cluster 1
Top features: ['interior', 'seat', 'mileage', 'comfortable', 'gas', 'gas mileage', 'transmission', 'car', 'performance', 'quality']
Reviews file names : ['/content/drive/MyDrive/ESAA/opinosis+opinion+frasl+review/OpinosisDataset1', '/content/drive/MyDrive/ESAA/opinosis+opinion+frasl